# Gemma 4 Particle Edu — Ollama Live Demo

This notebook demonstrates **Gemma 4 running locally via Ollama** on a Kaggle GPU.
It runs the same 5-step DAG pipeline used in the web app, proving the system works end-to-end with Ollama.

**What this proves:**
- Gemma 4 runs via Ollama on a single GPU (no cloud API)
- The DAG pipeline (Analyze → Research → Design → Generate → Validate) produces valid simulation JSON
- Physics parameters match SI reference values
- Zero API cost, fully local

**Companion demo — Gemma 4 via Google AI Studio**: The web app also supports running the same pipeline through Gemma 4 31B on Google AI Studio (free tier, 15 RPM), with `lookup_material()` function calling against a 138-material SI database. Activated via `?model=gemma4` URL parameter. See the companion writeup notebook for details.

In [ ]:
%%bash
# Install Ollama
curl -fsSL https://ollama.com/install.sh | sh
echo "Ollama installed"

In [ ]:
%%bash
# Start Ollama server in background and pull Gemma 4
nohup ollama serve > /tmp/ollama.log 2>&1 &
sleep 3

# Pull the smallest Gemma 4 model that fits Kaggle GPU
ollama pull gemma4
echo "Model ready"
ollama list

In [ ]:
import requests
import json
import time

OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL = 'gemma4'

def ollama_chat(system_prompt, user_prompt, temperature=0.7):
    """Send a chat request to local Ollama and return the full response."""
    start = time.time()
    resp = requests.post(OLLAMA_URL, json={
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        'stream': False,
        'options': {'temperature': temperature}
    })
    elapsed = time.time() - start
    data = resp.json()
    text = data.get('message', {}).get('content', '')
    return text, elapsed

# Test connection
tags = requests.get('http://localhost:11434/api/tags').json()
print(f"Ollama models: {[m['name'] for m in tags.get('models', [])]}")
print(f"Server: http://localhost:11434")

## 5-Step DAG Pipeline Demo

This is the exact same pipeline that runs in the web app.
Each step's output feeds into the next step.

In [ ]:
# ═══════════════════════════════════════════
# DAG Pipeline: 5-step reasoning chain
# User request: "Simulate a DNA double helix"
# ═══════════════════════════════════════════

USER_REQUEST = "Simulate a DNA double helix with accurate molecular physics"
print(f"User request: {USER_REQUEST}")
print("=" * 60)

In [ ]:
# STEP 1: ANALYZE
print("\n[STEP 1: ANALYZE]")
step1, t1 = ollama_chat(
    'You are a physics simulation expert. Analyze the request concisely.',
    f'[STEP 1: ANALYZE]\nUser request: "{USER_REQUEST}"\n\n'
    'Identify:\n1. What object/phenomenon\n2. Key physical properties\n3. Science domain\n4. Scale\n\n'
    'Respond in 3-4 bullet points.'
)
print(f"({t1:.1f}s)")
print(step1[:500])

In [ ]:
# STEP 2: RESEARCH
print("\n[STEP 2: RESEARCH]")
step2, t2 = ollama_chat(
    'You are a physics reference expert. Provide EXACT SI values.',
    f'[STEP 2: RESEARCH]\nBased on:\n{step1[:300]}\n\n'
    'List EXACT SI physical values:\n'
    '- gravity (m/s2): Earth=-9.81, Moon=-1.62, space=0\n'
    '- density (kg/m3): DNA=1700, water=1000\n'
    '- temperature (K): body=310, room=293\n'
    '- springStiffness: soft=5, medium=20, rigid=50\n\n'
    'Be PRECISE.'
)
print(f"({t2:.1f}s)")
print(step2[:500])

In [ ]:
# STEP 3: DESIGN
print("\n[STEP 3: DESIGN]")
step3, t3 = ollama_chat(
    'You are a particle simulation designer.',
    f'[STEP 3: DESIGN]\nPhysical properties:\n{step2[:300]}\n\n'
    'Design the particle simulation:\n'
    '- How many particles? (15000-25000)\n'
    '- What shapes? (helix, sphere, line, ring, etc.)\n'
    '- Connections? (chain, grid, nearest:N)\n'
    'Describe briefly.'
)
print(f"({t3:.1f}s)")
print(step3[:500])

In [ ]:
# STEP 4: GENERATE
print("\n[STEP 4: GENERATE]")
step4, t4 = ollama_chat(
    'You are a simulation JSON generator. Output MUST include ```json block.',
    f'[STEP 4: GENERATE]\nAnalysis: {step1[:200]}\nValues: {step2[:200]}\nDesign: {step3[:200]}\n\n'
    'Generate FINAL simulation JSON:\n'
    '```json\n{"simulation":{"prompt":"custom","title":"...","domain":"...",'
    '"physics":{"gravity":0,"damping":0.99,"springStiffness":30,"particleCount":15000,'
    '"temperature":310,"density":1.7,"viscosity":0.5}}}\n```',
    temperature=0.3
)
print(f"({t4:.1f}s)")
print(step4[:800])

In [ ]:
# STEP 5: VALIDATE
print("\n[STEP 5: VALIDATE]")
step5, t5 = ollama_chat(
    'You are a physics QA validator. Check and return ```json block.',
    f'[STEP 5: VALIDATE]\nOriginal: "{USER_REQUEST}"\nGenerated:\n{step4}\n\n'
    'Check: gravity correct? density correct? temperature correct?\n'
    'If ALL correct: return same JSON. If wrong: return CORRECTED JSON.\n'
    'ALWAYS include ```json block.',
    temperature=0.1
)
print(f"({t5:.1f}s)")
print(step5[:800])

In [ ]:
# ═══════════════════════════════════════════
# RESULT: Extract and validate JSON
# ═══════════════════════════════════════════
import re

final = step5 or step4
match = re.search(r'```json\s*([\s\S]*?)```', final)

print("\n" + "=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Total time: {t1+t2+t3+t4+t5:.1f}s")
print(f"Step times: Analyze={t1:.1f}s, Research={t2:.1f}s, Design={t3:.1f}s, Generate={t4:.1f}s, Validate={t5:.1f}s")

if match:
    try:
        sim_json = json.loads(match.group(1))
        print(f"\nJSON parse: SUCCESS")
        sim = sim_json.get('simulation', {})
        physics = sim.get('physics', {})
        print(f"Title: {sim.get('title', 'N/A')}")
        print(f"Domain: {sim.get('domain', 'N/A')}")
        print(f"\nPhysics parameters:")
        for k, v in physics.items():
            print(f"  {k}: {v}")
        
        # Validate key values for DNA
        print(f"\nValidation:")
        g = physics.get('gravity', -9.81)
        print(f"  gravity={g} {'PASS (zero-g for molecular)' if abs(g) < 1 else 'WARN (expected ~0)'}")
        t = physics.get('temperature', 293)
        print(f"  temperature={t}K {'PASS (body temp)' if 300 <= t <= 320 else 'WARN (expected ~310K)'}")
        
    except json.JSONDecodeError as e:
        print(f"JSON parse: FAILED ({e})")
else:
    print("No ```json block found in response")

## Run 10 Diverse Scenarios

Quick validation across multiple physics domains.

In [ ]:
SCENARIOS = [
    ("Build the Great Pyramid of Giza", {"gravity": -9.81, "material": "limestone"}),
    ("Simulate Moon surface gravity", {"gravity": -1.62}),
    ("Show water wave dynamics", {"gravity": -9.81, "material": "water"}),
    ("DNA double helix at body temperature", {"gravity": 0, "temperature": 310}),
    ("Earthquake test on steel building", {"gravity": -9.81, "material": "steel"}),
    ("Solar surface plasma", {"gravity": -274, "temperature": 5778}),
    ("Zero gravity space station", {"gravity": 0}),
    ("Tornado wind simulation", {"gravity": -9.81}),
    ("Crystal lattice structure", {"gravity": 0}),
    ("MOSFET transistor electron flow", {"gravity": 0}),
]

results = []
for i, (prompt, expected) in enumerate(SCENARIOS):
    print(f"\n[{i+1}/10] {prompt}")
    # Single-call mode (faster for batch)
    resp, elapsed = ollama_chat(
        'Generate simulation JSON. Include ```json block with {"simulation":{"prompt":"...","physics":{...}}}.',
        prompt,
        temperature=0.3
    )
    
    match = re.search(r'```json\s*([\s\S]*?)```', resp)
    status = 'FAIL'
    if match:
        try:
            parsed = json.loads(match.group(1))
            physics = parsed.get('simulation', {}).get('physics', {})
            status = 'PASS'
            # Check expected values
            for key, exp_val in expected.items():
                if key == 'material': continue
                actual = physics.get(key)
                if actual is not None and abs(actual - exp_val) > abs(exp_val) * 0.5 + 1:
                    status = 'WARN'
        except:
            status = 'PARSE_ERROR'
    
    results.append({'prompt': prompt, 'status': status, 'time': elapsed})
    print(f"  {status} ({elapsed:.1f}s)")

passed = sum(1 for r in results if r['status'] == 'PASS')
warned = sum(1 for r in results if r['status'] == 'WARN')
failed = sum(1 for r in results if r['status'] in ('FAIL', 'PARSE_ERROR'))
avg_time = sum(r['time'] for r in results) / len(results)

print(f"\n{'='*60}")
print(f"BATCH RESULTS: {passed} PASS, {warned} WARN, {failed} FAIL")
print(f"Average response time: {avg_time:.1f}s")
print(f"Total time: {sum(r['time'] for r in results):.0f}s")
print(f"Model: {MODEL} (Ollama local, Kaggle GPU)")

## Conclusion

This notebook demonstrates that Gemma 4 runs locally via Ollama on a single GPU and produces valid physics simulation parameters through the 5-step DAG pipeline.

- Zero cloud API cost
- All processing happens locally
- Same pipeline as the web application
- Student data never leaves the machine
- 48 E2E tests passing (including a 30-second pyramid structural drift test — 0.1% drift)

**Links:**
- Web Demo (Gemini default): https://gemma4-particle-edu.vercel.app
- Web Demo (Gemma 4 via AI Studio): https://gemma4-particle-edu.vercel.app/?model=gemma4
- GitHub: https://github.com/U2SY26/gemma4-particle-edu
- Video: https://youtu.be/3e-LZPHBA2M
- Companion writeup: https://www.kaggle.com/code/syu21125/gemma-4-particle-edu-3d-physics-simulation
- Benchmark dataset: https://www.kaggle.com/datasets/syu21125/gemma4-particle-edu-benchmark-300